In [ ]:
!pip install beamngpy

In [ ]:
from beamngpy import BeamNGpy, Scenario, Vehicle
from beamngpy.sensors import Camera, RoadsSensor

In [ ]:
# create BeamNGpy scenario
beamng = BeamNGpy("localhost", 25252, home='C:/BeamNG.tech.v0.38.3.0/')
beamng.open(launch=True)

In [ ]:
scenario = Scenario(
    "hirochi_raceway",
    "ai_driving",
    description="This is a test to load up Hirochi Raceway"
)

sbr = Vehicle(
    "sbr",
    model="sbr",
    license="RED",
    color="Red",
    part_config="vehicles/sbr/TT_AWD_S_DCT.pc"
)

scenario.add_vehicle(
    sbr,
    pos=(-408.48, 260.23, 25.14),       # Hirochi Raceway Starting line
    rot_quat=(0, 0, -0.2799, 0.9600)    # Facing starting line
)

scenario.make(beamng)

In [ ]:
# configure game state
beamng.settings.set_deterministic(60)

# load scenario and start
# beamng.control.pause()
beamng.scenario.load(scenario)
beamng.scenario.start()

In [ ]:
beamng.control.resume()

In [ ]:
# instantiate new roads sensor
roadsSensor = RoadsSensor(
    "roads1",
    beamng,
    sbr,
    is_visualised=True
)

camera1 = Camera(
    "camera1",
    beamng,
    sbr,
    requested_update_time=0.01,
    is_using_shared_memory=True,
    pos=(-0.3, 1, 2),
    dir=(0, -1, 0),
    field_of_view_y=90,
    near_far_planes=(0.1, 1000),
    resolution=(1024, 1024),
    is_streaming=True,
    is_render_annotations=True
)

In [ ]:
# TODO: Identify which road network is the short circuit
all_roads = beamng.scenario.get_road_network()
center_line_waypoints = []
#for x in all_roads[22544.0]["edges"]:
#    center_line_waypoints.append(x["middle"])
for x in beamng.scenario.get_road_network().keys():
    print(all_roads[x])


In [ ]:
print(beamng.scenario.get_roads().keys())

In [ ]:
import plotly.graph_objects as go

# 1. Unpack the coordinates
x_vals = [point[0] for point in center_line_waypoints]
y_vals = [point[1] for point in center_line_waypoints]
z_vals = [point[2] for point in center_line_waypoints]

# 2. Create the interactive 3D line plot
fig = go.Figure(data=go.Scatter3d(
    x=x_vals,
    y=y_vals,
    z=z_vals,
    mode='lines+markers', # Shows the line AND the individual waypoints
    marker=dict(
        size=3,
        color=z_vals,                # Color the points based on their elevation
        colorscale='Viridis',        # Use a cool color gradient
        opacity=0.8
    ),
    line=dict(
        color='darkblue',
        width=4
    )
))

# 3. Add titles and adjust the camera layout
fig.update_layout(
    title="Interactive BeamNG Track Spline",
    scene=dict(
        xaxis_title='X Axis',
        yaxis_title='Y Axis',
        zaxis_title='Z (Elevation)'
    )
)

# 4. Show the interactive graph
fig.show()